# Sparse Gaussian Process Tutorial
Approximate Gaussian Process regression with inducing points for scalable uncertainty quantification.

## Goal
Fit a sparse variational GP on a moderately large dataset, monitor the evidence lower bound (ELBO), and visualise predictive uncertainty.

In [1]:
# Configure Python path so the notebook sees the local deepuq package
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd())
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = str(PROJECT_ROOT / 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

In [2]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from deepuq.models import SparseGaussianProcessRegressor

## Generate Data
We draw 400 noisy samples from a smooth target function and keep an even grid for evaluation.

In [3]:
torch.manual_seed(7)
np.random.seed(7)

def target_fn(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.5 * math.pi * x) + 0.5 * torch.cos(1.5 * math.pi * x)

n_train = 400
x_train = torch.linspace(-1.2, 1.2, n_train).unsqueeze(-1)
y_clean = target_fn(x_train)
noise = 0.1 * torch.randn_like(y_clean)
y_train = y_clean + noise

x_test = torch.linspace(-1.5, 1.5, 300).unsqueeze(-1)
y_test = target_fn(x_test)

x_train[:5], y_train[:5]

(tensor([[-1.2000],
         [-1.1940],
         [-1.1880],
         [-1.1820],
         [-1.1759]]),
 tensor([[0.3225],
         [0.3884],
         [0.3828],
         [0.0980],
         [0.1641]]))

## Train Sparse Variational GP
We use 40 inducing points and optimise the ELBO with Adam. The training curve confirms convergence.

In [4]:
gp = SparseGaussianProcessRegressor(
    num_inducing=40,
    learning_rate=5e-2,
    num_iterations=600,
    verbose=False,
)
gp.fit(x_train, y_train)
print(f'Learned lengthscale: {gp.lengthscale_.item():.3f}')
print(f'Learned outputscale: {gp.outputscale_.item():.3f}')
print(f'Learned noise variance: {gp.noise_.item():.4f}')

_LinAlgError: linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 40 is not positive-definite).

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(gp.elbo_history)
plt.xlabel('Iteration')
plt.ylabel('ELBO')
plt.title('Sparse GP training curve')
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.4)
plt.show()

## Posterior Prediction
Sample the posterior mean and uncertainty bands on the evaluation grid.

In [ ]:
with torch.no_grad():
    mean, var = gp.predict(x_test)
    samples = gp.posterior_samples(x_test, n_samples=5)
mean = mean.cpu()
std = var.clamp_min(0.0).sqrt().cpu()
samples = samples.cpu()
x_test_np = x_test.squeeze(-1).cpu().numpy()
x_train_np = x_train.squeeze(-1).cpu().numpy()
y_train_np = y_train.squeeze(-1).cpu().numpy()
y_test_np = y_test.squeeze(-1).cpu().numpy()


In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(x_train_np, y_train_np, color='tab:blue', s=15, alpha=0.5, label='Train observations')
plt.plot(x_test_np, y_test_np, color='tab:green', linestyle='--', label='True function')
plt.plot(x_test_np, mean.numpy(), color='tab:red', label='Posterior mean')
plt.fill_between(x_test_np, mean.numpy() - 2 * std.numpy(), mean.numpy() + 2 * std.numpy(),
                 color='tab:red', alpha=0.15, label='95% credible interval')
for idx, sample in enumerate(samples):
    plt.plot(x_test_np, sample.numpy(), color='tab:orange', alpha=0.25, linewidth=1.0, label='Posterior sample' if idx == 0 else None)
plt.legend(frameon=False)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sparse GP posterior predictive')
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.4)
plt.show()

## Quantitative Metrics
Evaluate the posterior mean RMSE and the average predictive uncertainty.

In [ ]:
rmse = torch.sqrt(torch.mean((mean - y_test.squeeze(-1).cpu()) ** 2))
avg_std = std.mean()
print(f'Posterior mean RMSE: {rmse:.4f}')
print(f'Average predictive std: {avg_std:.4f}')